![Banner Eventos](https://www.camara.leg.br/internet/deputado/img/logo-camara.png)

# 📅 Fast Track — 03. Bronze: Eventos

**Ingestão de Dados Brutos — API Câmara dos Deputados**

| Item | Descrição |
|---|---|
| **Objetivo** | Ingerir dados brutos de eventos da API da Câmara dos Deputados para a camada Bronze |
| **Entidade** | Eventos (sessões plenárias, reuniões, audiências públicas, seminários) |
| **Endpoint API** | `https://dadosabertos.camara.leg.br/api/v2/eventos` |
| **Tabela Destino** | `uc_fast_track.bronze.eventos_raw` |
| **Modo** | APPEND (histórico completo de todas as execuções) |
| **Checkpoint** | Por `id` do evento (último ID processado) |
| **Formato Bronze** | Payload JSON raw (STRING) + colunas técnicas |
| **Responsável** | Ernesto Bassoli |
| **Programa** | Upskill Tiller – Engenharia de Dados (T2.7) |

---

## 📋 O Que Este Notebook Faz

Este notebook realiza a ingestão da entidade **Eventos** (camada Bronze):

1. **Recupera o Load ID** gerado pelo notebook 00_init_run
2. **Busca o último checkpoint** (último ID de evento processado)
3. **Chama a API** da Câmara dos Deputados com paginação automática
4. **Salva o payload JSON raw** em `bronze.eventos_raw` com colunas técnicas
5. **Registra logs detalhados** em `log.bronze_ingest_runs` e `ops.ingestion_requests`
6. **Atualiza checkpoint** para próxima execução incremental
7. **Valida** os dados ingeridos (contagem, amostra)

---

## 🎯 O Que São Eventos?

**Eventos** são atividades legislativas oficiais da Câmara dos Deputados:

**Tipos principais:**
* **Sessões Plenárias**: Reuniões do plenário (deliberativas, não deliberativas, solenes)
* **Reuniões de Comissões**: Encontros de comissões permanentes e temporárias
* **Audiências Públicas**: Debates abertos com convidados externos
* **Seminários**: Eventos de discussão temática
* **Outros**: Homenagens, lançamentos, cerimônias

**Características:**
* Têm data/hora de início e fim
* Podem ter situação: Convocada, Em Andamento, Encerrada, Cancelada
* Associados a órgãos (plenário, comissões)
* Podem ter requerimentos associados
* Importantes para análise de **agenda legislativa** e **participação parlamentar**

---

## 🔄 Quando Este Notebook É Executado

* **Workflow A (Diário)**: Task A03_bronze_eventos
* **Workflow B (Micro-batch 10 min)**: Task B03_bronze_eventos (captura eventos em tempo real)
* **Manualmente**: Para reprocessamento ou testes

---

## 🎯 Entregas Deste Notebook

Ao final da execução:

✅ **Dados Ingeridos**: N registros novos em `bronze.eventos_raw`
✅ **Checkpoint Atualizado**: Último ID processado salvo em `ops.checkpoints`
✅ **Logs Registrados**: 1 linha em `log.bronze_ingest_runs` com métricas
✅ **Requests Logados**: N linhas em `ops.ingestion_requests` (1 por chamada HTTP)
✅ **Validação**: Contagem e amostra dos dados exibidos

## ⚙️ Bloco 1: Configuração e Recuperação do Load ID

### 📋 Descrição

Este bloco recupera as **variáveis globais e o Load ID** criados pelo notebook `00_init_run`.

---

### 🔗 Dependência do Notebook 00

Este notebook **depende** do notebook `00_init_run` ter sido executado primeiro.

**Variáveis necessárias:**
* `load_id`: UUID único que identifica esta execução
* `env`: Ambiente (dev/hml/prd)
* `run_date`: Data de referência desta execução
* `full_refresh`: Se deve reprocessar tudo ou apenas incremental
* `catalog`: Nome do catálogo UC (`uc_fast_track`)
* `schema_bronze`, `schema_ops`, `schema_log`: Nomes dos schemas

In [0]:
# ============================================================
# RECUPERAÇÃO DE VARIÁVEIS DO NOTEBOOK 00_init_run
# ============================================================

print("🔗 Recuperando variáveis do notebook 00_init_run...")
print()

# Verificação: Load ID Disponível?
try:
    print(f"✅ Load ID encontrado: {load_id}")
    print(f"   Este UUID identifica esta execução específica")
except NameError:
    print("❌ ERRO: Load ID não encontrado!")
    print("   Execute o notebook 00_init_run primeiro")
    raise Exception("Load ID não disponível - execute notebook 00 primeiro")

print()

# Recuperação: Outras Variáveis Globais
try:
    print(f"✅ Ambiente (env):              {env}")
    print(f"✅ Data de Execução (run_date): {run_date}")
    print(f"✅ Full Refresh:                {full_refresh}")
    print()
    print(f"✅ Catálogo UC:                 {catalog}")
    print(f"✅ Schema Bronze:               {schema_bronze}")
    print(f"✅ Schema Ops:                  {schema_ops}")
    print(f"✅ Schema Log:                  {schema_log}")
    print()
except NameError as e:
    print(f"❌ ERRO: Variável não encontrada: {e}")
    print("   Execute o notebook 00_init_run primeiro")
    raise

# Definição: Variáveis Específicas deste Notebook
entity_name = "eventos"
api_base_url = "https://dadosabertos.camara.leg.br/api/v2"
api_endpoint = f"{api_base_url}/eventos"
table_bronze = f"{catalog}.{schema_bronze}.{entity_name}_raw"

# Configurações de paginação
page_size = 100
max_retries = 3
retry_delay_seconds = 5

print("✅ Variáveis específicas definidas:")
print(f"   • Entidade:           {entity_name}")
print(f"   • Endpoint API:       {api_endpoint}")
print(f"   • Tabela Destino:     {table_bronze}")
print(f"   • Page Size:          {page_size}")
print()
print("="*70)

## 📦 Bloco 2: Importações Específicas

Importações necessárias para ingestão Bronze de eventos.

In [0]:
# ============================================================
# IMPORTAÇÕES ESPECÍFICAS PARA INGESTÃO BRONZE
# ============================================================

print("📦 Verificando bibliotecas...")

import requests
import json
import time
import uuid
import hashlib
from datetime import datetime

from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DateType
from pyspark.sql.functions import col, lit, current_timestamp, sha2

print("✅ Bibliotecas disponíveis")
print()

## 🔌 Bloco 3: Função de Extração da API

### 📋 Descrição

Define a função `get_eventos_from_api()` que:
1. Chama o endpoint `/eventos` da API
2. Trata paginação automática
3. Implementa retry com backoff exponencial
4. Loga cada request HTTP
5. Retorna lista de payloads JSON

---

### 🌐 Endpoint da API

**URL:** `https://dadosabertos.camara.leg.br/api/v2/eventos`

**Parâmetros:**
* `pagina`: Número da página
* `itens`: Registros por página (máx 100)
* `ordem`: ASC/DESC
* `ordenarPor`: Campo para ordenação
* `dataInicio`, `dataFim`: Filtros de período (opcional)

**Resposta:**
```json
{
  "dados": [
    {"id": 67890, "dataHoraInicio": "2026-04-27T14:00", "descricao": "Sessão Deliberativa Ordinária", ...},
    {"id": 67891, "dataHoraInicio": "2026-04-27T15:00", "descricao": "Reunião CCJ", ...}
  ],
  "links": [{"rel": "next", "href": "..."}]
}
```

In [0]:
# ============================================================
# FUNÇÃO: EXTRAÇÃO DE EVENTOS DA API
# ============================================================

def get_eventos_from_api(checkpoint_id=None):
    """
    Extrai lista de eventos da API da Câmara dos Deputados.
    
    Args:
        checkpoint_id (int, optional): Último ID processado.
    
    Returns:
        list: Lista de dicts com payloads JSON dos eventos.
    """
    
    print(f"🔌 Iniciando extração da API: {api_endpoint}")
    print()
    
    all_eventos = []
    pagina_atual = 1
    has_more_pages = True
    total_requests = 0
    total_records = 0
    
    while has_more_pages:
        print(f"📄 Processando página {pagina_atual}...")
        
        params = {
            "pagina": pagina_atual,
            "itens": page_size,
            "ordem": "ASC",
            "ordenarPor": "id"
        }
        
        request_id = str(uuid.uuid4())
        request_start_time = time.time()
        
        success = False
        response_data = None
        http_status = None
        error_message = None
        
        for attempt in range(1, max_retries + 1):
            try:
                response = requests.get(api_endpoint, params=params, timeout=30)
                http_status = response.status_code
                
                if response.status_code == 200:
                    response_data = response.json()
                    success = True
                    break
                else:
                    error_message = f"HTTP {response.status_code}"
                    print(f"   ⚠️  Tentativa {attempt}/{max_retries} falhou: {error_message}")
                    if attempt < max_retries:
                        time.sleep(retry_delay_seconds * attempt)
            
            except requests.exceptions.RequestException as e:
                error_message = str(e)
                print(f"   ⚠️  Tentativa {attempt}/{max_retries} falhou: {error_message}")
                if attempt < max_retries:
                    time.sleep(retry_delay_seconds * attempt)
        
        response_time_ms = int((time.time() - request_start_time) * 1000)
        total_requests += 1
        
        # Loga request HTTP
        log_request_data = [{
            "request_id": request_id,
            "load_id": load_id,
            "entity_name": entity_name,
            "endpoint": f"{api_endpoint}?pagina={pagina_atual}",
            "http_method": "GET",
            "http_status": http_status,
            "response_time_ms": response_time_ms,
            "success": success,
            "error_message": error_message,
            "request_timestamp": datetime.now()
        }]
        
        df_log_request = spark.createDataFrame(log_request_data)
        df_log_request.write.mode("append").saveAsTable(f"{catalog}.{schema_ops}.ingestion_requests")
        
        if not success:
            print(f"   ❌ Falha após {max_retries} tentativas. Abortando.")
            raise Exception(f"Falha ao chamar API: {error_message}")
        
        eventos_page = response_data.get("dados", [])
        records_in_page = len(eventos_page)
        print(f"   ✅ {records_in_page} registros recebidos ({response_time_ms}ms)")
        
        # Filtrar por checkpoint
        if checkpoint_id is not None:
            eventos_page = [e for e in eventos_page if e.get("id", 0) > checkpoint_id]
            print(f"   🔍 Filtrados por checkpoint (id > {checkpoint_id}): {len(eventos_page)} novos")
        
        all_eventos.extend(eventos_page)
        total_records += len(eventos_page)
        
        links = response_data.get("links", [])
        has_next = any(link.get("rel") == "next" for link in links)
        
        if has_next and len(eventos_page) > 0:
            pagina_atual += 1
            time.sleep(0.5)
        else:
            has_more_pages = False
            print(f"   ℹ️  Última página alcançada")
        
        print()
    
    print("="*70)
    print(f"✅ Extração concluída")
    print(f"   • Total de requests HTTP: {total_requests}")
    print(f"   • Total de registros extraídos: {total_records}")
    print("="*70)
    print()
    
    return all_eventos

print("✅ Função get_eventos_from_api() definida")
print()

## 🔖 Bloco 4: Checkpoint e Modo Incremental

Busca o **último checkpoint** para processamento incremental.

**Checkpoint para eventos:** `checkpoint_type = 'last_id'` → Último ID de evento processado

In [0]:
# ============================================================
# BUSCAR ÚLTIMO CHECKPOINT
# ============================================================

print("🔖 Buscando último checkpoint...")
print()

if full_refresh:
    print("🔄 Modo: FULL REFRESH")
    print("   Checkpoint será ignorado - reprocessando tudo")
    checkpoint_value = None
else:
    print("📈 Modo: INCREMENTAL")
    print("   Buscando último checkpoint salvo...")
    print()
    
    query_checkpoint = f"""
    SELECT checkpoint_value, updated_at
    FROM {catalog}.{schema_ops}.checkpoints
    WHERE source_endpoint = '/eventos'
      AND checkpoint_type = 'last_id'
    ORDER BY updated_at DESC
    LIMIT 1
    """
    
    try:
        df_checkpoint = spark.sql(query_checkpoint)
        
        if df_checkpoint.count() > 0:
            checkpoint_row = df_checkpoint.first()
            checkpoint_value = int(checkpoint_row["checkpoint_value"])
            checkpoint_updated_at = checkpoint_row["updated_at"]
            
            print(f"   ✅ Checkpoint encontrado:")
            print(f"      • Último ID processado: {checkpoint_value}")
            print(f"      • Atualizado em:        {checkpoint_updated_at}")
            print(f"      • Processará IDs > {checkpoint_value}")
        else:
            print("   ℹ️  Nenhum checkpoint encontrado (primeira execução)")
            print("      Processará todos os registros")
            checkpoint_value = None
    
    except Exception as e:
        print(f"   ⚠️  Erro ao buscar checkpoint: {e}")
        print("      Processará todos os registros (modo seguro)")
        checkpoint_value = None

print()
print("="*70)
print(f"📌 Checkpoint definido: {checkpoint_value}")
if checkpoint_value is None:
    print("   → Processará TODOS os eventos")
else:
    print(f"   → Processará apenas eventos com ID > {checkpoint_value}")
print("="*70)
print()

## 🗄️ Bloco 5: Criação da Tabela Bronze

Cria a tabela `bronze.eventos_raw` se não existir.

**Schema:** payload (STRING) + colunas técnicas (record_id, record_hash, load_id, timestamps, env)

In [0]:
# ============================================================
# CRIAÇÃO DA TABELA BRONZE: eventos_raw
# ============================================================

print(f"🗄️  Criando tabela Bronze: {table_bronze}...")
print()

sql_create_table = f"""
CREATE TABLE IF NOT EXISTS {table_bronze} (
    payload STRING COMMENT 'Payload JSON raw completo da API (sem transformações)',
    record_id STRING COMMENT 'ID único do evento (evento.id extraído do JSON)',
    record_hash STRING COMMENT 'Hash SHA256 do payload (identifica mudanças)',
    load_id STRING COMMENT 'UUID único da execução que ingeriu este registro',
    ingestion_timestamp TIMESTAMP COMMENT 'Timestamp de quando foi ingerido',
    ingestion_date DATE COMMENT 'Data de ingestão (particionamento)',
    source_endpoint STRING COMMENT 'Endpoint da API de origem',
    env STRING COMMENT 'Ambiente de execução (dev/hml/prd)'
)
USING DELTA
PARTITIONED BY (ingestion_date)
COMMENT 'Tabela Bronze - Eventos Raw (sessões, reuniões, audiências - payload JSON completo)'
"""

spark.sql(sql_create_table)

print(f"✅ Tabela '{table_bronze}' criada/validada")
print(f"   • Formato: Delta Lake")
print(f"   • Particionamento: ingestion_date")
print(f"   • Mode: APPEND")
print()
print("="*70)
print()

## 📥 Bloco 6: Ingestão - Extração e Enriquecimento

Chama a API, enriquece payloads com colunas técnicas e cria DataFrame.

In [0]:
# ============================================================
# INGESTÃO: EXTRAIR DADOS DA API
# ============================================================

print("📥 Iniciando ingestão de eventos...")
print()

ingest_start_time = time.time()

# Chamar API
eventos_list = get_eventos_from_api(checkpoint_id=checkpoint_value)
num_records_extracted = len(eventos_list)

print(f"✅ {num_records_extracted} eventos extraídos da API")
print()

# Verificar se há dados
if num_records_extracted == 0:
    print("⚠️  Nenhum registro novo encontrado")
    print("   Finalizando execução")
    print()
    
    log_bronze_data = [{
        "load_id": load_id,
        "entity_name": entity_name,
        "records_extracted": 0,
        "records_ingested": 0,
        "api_calls_count": 0,
        "checkpoint_before": str(checkpoint_value) if checkpoint_value else None,
        "checkpoint_after": str(checkpoint_value) if checkpoint_value else None,
        "execution_time_sec": int(time.time() - ingest_start_time),
        "status": "NO_NEW_DATA",
        "run_date": run_date,
        "env": env,
        "created_at": datetime.now()
    }]
    
    spark.createDataFrame(log_bronze_data).write.mode("append").saveAsTable(
        f"{catalog}.{schema_log}.bronze_ingest_runs"
    )
    
    dbutils.notebook.exit("NO_NEW_DATA")

# Enriquecer com colunas técnicas
print("🔧 Enriquecendo payloads...")

enriched_records = []
current_ts = datetime.now()
current_date = current_ts.date()

for evento_payload in eventos_list:
    payload_json_str = json.dumps(evento_payload, ensure_ascii=False)
    payload_hash = hashlib.sha256(payload_json_str.encode('utf-8')).hexdigest()
    record_id = str(evento_payload.get("id", "unknown"))
    
    enriched_record = {
        "payload": payload_json_str,
        "record_id": record_id,
        "record_hash": payload_hash,
        "load_id": load_id,
        "ingestion_timestamp": current_ts,
        "ingestion_date": current_date,
        "source_endpoint": "/eventos",
        "env": env
    }
    enriched_records.append(enriched_record)

print(f"✅ {len(enriched_records)} registros enriquecidos")
print()

# Converter para DataFrame
print("🔄 Convertendo para DataFrame Spark...")

schema_bronze = StructType([
    StructField("payload", StringType(), True),
    StructField("record_id", StringType(), True),
    StructField("record_hash", StringType(), True),
    StructField("load_id", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("ingestion_date", DateType(), True),
    StructField("source_endpoint", StringType(), True),
    StructField("env", StringType(), True)
])

df_bronze = spark.createDataFrame(enriched_records, schema=schema_bronze)

print(f"✅ DataFrame criado com {df_bronze.count()} registros")
print()
print("="*70)
print()

## 💾 Bloco 7: Salvar em Bronze (APPEND)

Persiste o DataFrame em `bronze.eventos_raw` mantendo histórico completo.

In [0]:
# ============================================================
# SALVAR DATAFRAME EM BRONZE (MODE APPEND)
# ============================================================

print(f"💾 Salvando dados em {table_bronze}...")
print()

write_start_time = time.time()

try:
    df_bronze.write \
        .format("delta") \
        .mode("append") \
        .partitionBy("ingestion_date") \
        .saveAsTable(table_bronze)
    
    write_time_sec = int(time.time() - write_start_time)
    
    print(f"✅ Dados salvos com sucesso!")
    print(f"   • Tabela: {table_bronze}")
    print(f"   • Registros salvos: {num_records_extracted}")
    print(f"   • Tempo de escrita: {write_time_sec}s")
    print(f"   • Mode: APPEND")
    print()
    
    ingest_success = True
    ingest_error = None
    
except Exception as e:
    write_time_sec = int(time.time() - write_start_time)
    ingest_success = False
    ingest_error = str(e)
    print(f"❌ ERRO ao salvar: {e}")
    print()
    raise

finally:
    ingest_total_time_sec = int(time.time() - ingest_start_time)
    print("="*70)
    print(f"⏱️  Tempo total: {ingest_total_time_sec}s")
    print("="*70)
    print()

## 🔖 Bloco 8: Atualizar Checkpoint

Salva o maior ID ingerido em `ops.checkpoints` para próximas execuções incrementais.

In [0]:
# ============================================================
# ATUALIZAR CHECKPOINT
# ============================================================

print("🔖 Atualizando checkpoint...")
print()

max_id_row = df_bronze.selectExpr("CAST(record_id AS INT) as id_int") \
    .agg({"id_int": "max"}) \
    .first()

new_checkpoint_value = max_id_row[0]

print(f"📊 Checkpoint calculado:")
print(f"   • Checkpoint anterior: {checkpoint_value}")
print(f"   • Novo checkpoint:     {new_checkpoint_value}")
print()

if new_checkpoint_value is None:
    print("⚠️  Nenhum ID válido - checkpoint não será atualizado")
    print()
else:
    sql_merge_checkpoint = f"""
    MERGE INTO {catalog}.{schema_ops}.checkpoints AS target
    USING (
        SELECT 
            '/eventos' AS source_endpoint,
            'last_id' AS checkpoint_type,
            '{new_checkpoint_value}' AS checkpoint_value,
            current_timestamp() AS updated_at
    ) AS source
    ON target.source_endpoint = source.source_endpoint
       AND target.checkpoint_type = source.checkpoint_type
    WHEN MATCHED THEN 
        UPDATE SET 
            target.checkpoint_value = source.checkpoint_value,
            target.updated_at = source.updated_at
    WHEN NOT MATCHED THEN
        INSERT (source_endpoint, checkpoint_type, checkpoint_value, updated_at)
        VALUES (source.source_endpoint, source.checkpoint_type, source.checkpoint_value, source.updated_at)
    """
    
    spark.sql(sql_merge_checkpoint)
    
    print(f"✅ Checkpoint atualizado!")
    print(f"   • Endpoint:     /eventos")
    print(f"   • Tipo:         last_id")
    print(f"   • Valor:        {new_checkpoint_value}")
    print(f"   • Próxima exec: Processará IDs > {new_checkpoint_value}")
    print()

print("="*70)
print()

## 📝 Bloco 9: Logging Detalhado

Registra métricas em `log.bronze_ingest_runs` para auditoria e troubleshooting.

In [0]:
# ============================================================
# REGISTRAR LOG EM log.bronze_ingest_runs
# ============================================================

print("📝 Registrando log...")
print()

query_api_calls = f"""
SELECT COUNT(*) as api_calls
FROM {catalog}.{schema_ops}.ingestion_requests
WHERE load_id = '{load_id}'
  AND entity_name = '{entity_name}'
"""

api_calls_count = spark.sql(query_api_calls).first()[0]

log_bronze_data = [{
    "load_id": load_id,
    "entity_name": entity_name,
    "records_extracted": num_records_extracted,
    "records_ingested": num_records_extracted,
    "api_calls_count": api_calls_count,
    "checkpoint_before": str(checkpoint_value) if checkpoint_value else None,
    "checkpoint_after": str(new_checkpoint_value) if new_checkpoint_value else None,
    "execution_time_sec": ingest_total_time_sec,
    "status": "SUCCESS" if ingest_success else "FAILED",
    "error_message": ingest_error,
    "run_date": run_date,
    "env": env,
    "created_at": datetime.now()
}]

df_log = spark.createDataFrame(log_bronze_data)
df_log.write.mode("append").saveAsTable(f"{catalog}.{schema_log}.bronze_ingest_runs")

print(f"✅ Log registrado em {catalog}.{schema_log}.bronze_ingest_runs")
print(f"   • Load ID:       {load_id}")
print(f"   • Entidade:      {entity_name}")
print(f"   • Registros:     {num_records_extracted}")
print(f"   • API calls:     {api_calls_count}")
print(f"   • Tempo (s):     {ingest_total_time_sec}")
print(f"   • Status:        SUCCESS")
print()
print("="*70)
print()

## ✅ Bloco 10: Validação dos Dados

Valida que os dados foram salvos corretamente:
1. Contagem total
2. Contagem deste load
3. Amostra visual
4. Estatísticas por partição

In [0]:
# ============================================================
# VALIDAÇÃO DOS DADOS INGERIDOS
# ============================================================

print("✅ Validando dados...")
print()
print("="*70)

# Validação 1: Contagem Total
query_count_total = f"""
SELECT COUNT(*) as total_records
FROM {table_bronze}
"""
total_records = spark.sql(query_count_total).first()[0]

print(f"📊 VALIDAÇÃO 1: Contagem Total")
print(f"   • Total de registros: {total_records:,}")
print()

# Validação 2: Contagem Deste Load
query_count_this_load = f"""
SELECT COUNT(*) as records_this_load
FROM {table_bronze}
WHERE load_id = '{load_id}'
"""
records_this_load = spark.sql(query_count_this_load).first()[0]

print(f"📊 VALIDAÇÃO 2: Contagem Deste Load")
print(f"   • Ingeridos nesta execução: {records_this_load:,}")
print(f"   • Esperado: {num_records_extracted:,}")

if records_this_load == num_records_extracted:
    print(f"   ✅ Contagem OK!")
else:
    print(f"   ⚠️  Divergência na contagem!")
print()

# Validação 3: Amostra
print(f"📊 VALIDAÇÃO 3: Amostra (5 primeiros)")
print()

query_sample = f"""
SELECT 
    record_id,
    LEFT(record_hash, 12) as hash_preview,
    ingestion_timestamp,
    ingestion_date,
    env,
    LEFT(payload, 100) as payload_preview
FROM {table_bronze}
WHERE load_id = '{load_id}'
ORDER BY ingestion_timestamp
LIMIT 5
"""
df_sample = spark.sql(query_sample)
display(df_sample)

print()

# Validação 4: Estatísticas por Partição
print(f"📊 VALIDAÇÃO 4: Estatísticas por Partição")
print()

query_partitions = f"""
SELECT 
    ingestion_date,
    COUNT(*) as records_count,
    COUNT(DISTINCT load_id) as distinct_loads
FROM {table_bronze}
GROUP BY ingestion_date
ORDER BY ingestion_date DESC
LIMIT 10
"""
df_partitions = spark.sql(query_partitions)
display(df_partitions)

print()
print("="*70)
print()
print("✅ VALIDAÇÃO CONCLUÍDA!")
print()
print(f"🎯 Resumo Final:")
print(f"   • Tabela:           {table_bronze}")
print(f"   • Total registros:  {total_records:,}")
print(f"   • Novos:            {records_this_load:,}")
print(f"   • Load ID:          {load_id}")
print(f"   • Checkpoint:       {new_checkpoint_value}")
print(f"   • Status:           SUCCESS ✅")
print()
print("="*70)
print()
print("🎉 Notebook 03_bronze_eventos concluído com sucesso!")